# Import Components & Setup Model

In [1]:
import os
import sys

# Mengambil path dari root proyek (1 tingkat di atas folder notebooks)
notebook_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(notebook_dir, ".."))

# Daftarkan root direktori ke dalam sys.path milik Python jika belum ada
if root_dir not in sys.path:
    sys.path.append(root_dir)

print(f"[INFO] Berhasil menambahkan root ke sys.path: {root_dir}")

[INFO] Berhasil menambahkan root ke sys.path: /home/kiana/Downloads/Hasil_Scraping_Pariwisata/travel-rec


In [5]:
import pandas as pd
import numpy as np
from datetime import datetime
from sentence_transformers import SentenceTransformer

from src.components.preprocessing import DataPreprocessor
from src.components.feature_engineering import FeatureEngineer
from src.components.recommender import SemanticRecommender
from src.evaluation.evaluator import RecommenderEvaluator

# 1. Load & Preprocess Data
df = pd.read_csv("/home/kiana/Downloads/Hasil_Scraping_Pariwisata/travel-rec/data/processed/dataset_rekomendasi_final.csv")
preprocessor = DataPreprocessor()
df = preprocessor.preprocess(df)

# 2. Feature Engineering (Tuning 1: Pseudo-sentence)
engineer = FeatureEngineer()
df = engineer.create_combined_features(df)

# 3. Load IBM Granite Model & Generate Offline Embeddings
model_name = "ibm-granite/granite-embedding-30m-english"
embedding_model = SentenceTransformer(model_name)
embeddings = embedding_model.encode(df["combined_features"].tolist(), convert_to_numpy=True)

print(f"[INFO] Embeddings generated with shape: {embeddings.shape}")

[INFO] Preprocessing completed successfully. Dataset Shape: (1294, 23)
[INFO] Tuning 1 Applied Successfully. Pseudo-sentence string formatted. Dataset Shape: (1294, 23)


Loading weights: 100%|██████████████████| 103/103 [00:00<00:00, 127.04it/s]


[INFO] Embeddings generated with shape: (1294, 384)


# Define Test Cases for Benchmarking

In [6]:
# Buat skenario test case berdasarkan data real untuk dievaluasi
test_cases = [
    {
        "city": "Kuala Lumpur",
        "preference": "aquarium family ocean fish marine life",
        "gt": ["Aquaria KLCC"],
        "datetime": datetime.strptime("2026-05-25 14:00:00", "%Y-%m-%d %H:%M:%S") # Jam buka normal
    },
    {
        "city": "Den Haag",
        "preference": "romantic dinner cozy cocktail french restaurant",
        "gt": ["Restaurant Basaal", "BY AMI Restaurant | Den Haag"],
        "datetime": datetime.strptime("2026-05-25 19:30:00", "%Y-%m-%d %H:%M:%S") # Jam malam
    }
]
print(f"Defined {len(test_cases)} evaluation test cases.")

Defined 2 evaluation test cases.


# Run Tuning Grid Search ($\alpha$ and Threshold)

In [7]:
# Kita uji coba kombinasi alpha untuk melihat pengaruh Rating Bias
alpha_candidates = [0.0, 0.1, 0.2, 0.5]
threshold_candidates = [0.50, 0.55, 0.60]

best_mrr = -1
best_params = {}

for alpha in alpha_candidates:
    for threshold in threshold_candidates:
        print(f"\n[TESTING] Alpha (Rating Bias): {alpha} | Min Threshold: {threshold}")

        # Inisialisasi Recommender dengan hyperparameter kandidat
        recommender = SemanticRecommender(
            df=df,
            embeddings=embeddings,
            similarity_matrix=None,
            model=embedding_model,
            alpha=alpha,
            min_threshold=threshold
        )

        # Jalankan Evaluator
        evaluator = RecommenderEvaluator(recommender)
        metrics = evaluator.run_benchmark_test(test_cases)

        # Tracking parameter terbaik berdasarkan Mean Reciprocal Rank (MRR)
        if metrics["mean_mrr"] > best_mrr:
            best_mrr = metrics["mean_mrr"]
            best_params = {"alpha": alpha, "threshold": threshold, "metrics": metrics}

print("\n==================================================")
print("BEST TUNING CONFIGURATION FOUND:")
print(best_params)
print("==================================================")


[TESTING] Alpha (Rating Bias): 0.0 | Min Threshold: 0.5
[START BENCHMARK] Evaluating system tuning performance...

================ BENCHMARK REPORT ================
Total Test Sceanarios  : 2
Mean Reciprocal Rank   : 1.0000
Precision @ 5          : 0.3000
Fallback Safety Triggers: 0 times activated

[TESTING] Alpha (Rating Bias): 0.0 | Min Threshold: 0.55
[START BENCHMARK] Evaluating system tuning performance...

================ BENCHMARK REPORT ================
Total Test Sceanarios  : 2
Mean Reciprocal Rank   : 1.0000
Precision @ 5          : 0.3000
Fallback Safety Triggers: 0 times activated

[TESTING] Alpha (Rating Bias): 0.0 | Min Threshold: 0.6
[START BENCHMARK] Evaluating system tuning performance...

================ BENCHMARK REPORT ================
Total Test Sceanarios  : 2
Mean Reciprocal Rank   : 1.0000
Precision @ 5          : 0.3000
Fallback Safety Triggers: 0 times activated

[TESTING] Alpha (Rating Bias): 0.1 | Min Threshold: 0.5
[START BENCHMARK] Evaluating system 